# 02 Silver Clean & Enrich -- Vision Zero LA

**Workspace:** CTS-Geo Analytics GBD  
**Lakehouse:** VisionZero_Lakehouse  

**Purpose:** Read raw Bronze tables, apply data-quality rules, classify crash
severity, add spatial enrichment (H3 indexing, ArcGIS GeoAnalytics geometry),
and produce the `silver_crashes` Delta table for downstream Gold analytics.

**Transformations:**
- Parse lat/lon to double, date_occ to date, time_occ to integer hour
- Filter invalid coordinates (null, zero, outside LA County bounding box)
- Deduplicate on dr_no
- Classify severity from MO codes (fatal, pedestrian, bicycle, severe, other)
- Add crash_year, crash_month, year_month, intersection columns
- H3 indexing at resolutions 8 and 9
- ArcGIS GeoAnalytics ST_Point geometry
- School-zone proximity tagging via GeoAnalytics buffer + spatial join

**Inputs:** `bronze_crashes`, `bronze_schools`  
**Output:** `silver_crashes` (partitioned by crash_year)

In [ ]:
%pip install h3

In [ ]:
import json
import h3
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType

LAKEHOUSE = "VisionZero_Lakehouse"

# ── Read bronze_crashes ────────────────────────────────────────────────────────
bronze = spark.read.format("delta").table(f"bronze_crashes")
print(f"bronze_crashes: {bronze.count():,} rows")
bronze.printSchema()

In [ ]:
# ── Data quality: parse types, filter coords, deduplicate ─────────────────────

# LA County bounding box
LAT_MIN, LAT_MAX = 33.5, 35.0
LON_MIN, LON_MAX = -119.0, -117.5

cleaned = (
    bronze
    # Parse numeric / temporal columns
    .withColumn("latitude", F.col("lat").cast(DoubleType()))
    .withColumn("longitude", F.col("lon").cast(DoubleType()))
    .withColumn("crash_date", F.to_date(F.col("date_occ")))
    .withColumn("crash_hour", F.substring(F.col("time_occ"), 1, 2).cast(IntegerType()))

    # Filter invalid coordinates
    .filter(F.col("latitude").isNotNull())
    .filter(F.col("longitude").isNotNull())
    .filter(F.col("latitude") != 0.0)
    .filter(F.col("longitude") != 0.0)
    .filter(F.col("latitude").between(LAT_MIN, LAT_MAX))
    .filter(F.col("longitude").between(LON_MIN, LON_MAX))

    # Deduplicate on dr_no
    .dropDuplicates(["dr_no"])
)

print(f"After quality filters: {cleaned.count():,} rows")

In [ ]:
# ── Classify severity from mocodes field ──────────────────────────────────────

@F.udf(StringType())
def classify_severity(mocodes, crm_cd_desc):
    """Derive crash severity from LAPD MO codes and crime description.

    Priority order: fatal > pedestrian > bicycle > severe > other
    """
    codes = set((mocodes or "").strip().split())
    desc = (crm_cd_desc or "").upper()

    if "0449" in codes or "KILLED" in desc:
        return "fatal"
    if "1920" in codes or "PEDESTRIAN" in desc:
        return "pedestrian"
    if "1822" in codes or "BICYCLE" in desc or "BIKE" in desc:
        return "bicycle"
    if "0448" in codes or "FELONY" in desc:
        return "severe"
    return "other"


cleaned = cleaned.withColumn(
    "severity",
    classify_severity(F.col("mocodes"), F.col("crm_cd_desc"))
)

cleaned.groupBy("severity").count().orderBy(F.desc("count")).show()

In [ ]:
# ── Computed columns ──────────────────────────────────────────────────────────

silver = (
    cleaned
    .withColumn("crash_year", F.year(F.col("crash_date")))
    .withColumn("crash_month", F.month(F.col("crash_date")))
    .withColumn("year_month", F.date_format(F.col("crash_date"), "yyyy-MM"))
    .withColumn(
        "intersection",
        F.when(
            F.col("cross_street").isNotNull() & (F.col("cross_street") != ""),
            F.concat(F.col("location"), F.lit(" / "), F.col("cross_street"))
        ).otherwise(F.col("location"))
    )
)

print("Computed columns added: crash_year, crash_month, year_month, intersection")

In [ ]:
# ── H3 indexing at resolution 8 and 9 ────────────────────────────────────────

@F.udf(StringType())
def lat_lng_to_h3_r8(lat, lng):
    """Convert lat/lng to H3 cell index at resolution 8 (~460 m edge)."""
    try:
        return h3.latlng_to_cell(float(lat), float(lng), 8)
    except Exception:
        return None


@F.udf(StringType())
def lat_lng_to_h3_r9(lat, lng):
    """Convert lat/lng to H3 cell index at resolution 9 (~174 m edge)."""
    try:
        return h3.latlng_to_cell(float(lat), float(lng), 9)
    except Exception:
        return None


silver = (
    silver
    .withColumn("h3_res8", lat_lng_to_h3_r8(F.col("latitude"), F.col("longitude")))
    .withColumn("h3_res9", lat_lng_to_h3_r9(F.col("latitude"), F.col("longitude")))
    .filter(F.col("h3_res9").isNotNull())
)

print(f"H3-indexed rows: {silver.count():,}")

In [ ]:
# ── ArcGIS GeoAnalytics: create ST_Point geometry column ─────────────────────

from geoanalytics_fabric.sql import functions as GA

silver = silver.withColumn(
    "geometry",
    GA.ST_Point("longitude", "latitude", 4326)
)

print("Geometry column added (EPSG:4326 ST_Point).")
silver.select("dr_no", "latitude", "longitude", "geometry").show(5, truncate=False)

In [ ]:
# ── ArcGIS GeoAnalytics: school-zone buffer analysis ─────────────────────────
#
# Create a 300 m buffer around every school, then spatial-join crashes to
# determine which ones occurred inside a school zone.

try:
    schools = spark.read.format("delta").table(f"bronze_schools")
    school_count = schools.count()
    print(f"bronze_schools loaded: {school_count:,} rows")

    # Identify lat/lon columns (may vary by dataset schema)
    school_cols = [c.lower() for c in schools.columns]
    lat_col = "latitude" if "latitude" in school_cols else "lat"
    lon_col = "longitude" if "longitude" in school_cols else "lon"

    # Cast to double if necessary
    schools = (
        schools
        .withColumn("school_lat", F.col(lat_col).cast(DoubleType()))
        .withColumn("school_lon", F.col(lon_col).cast(DoubleType()))
        .filter(F.col("school_lat").isNotNull())
        .filter(F.col("school_lon").isNotNull())
    )

    # Build 300 m buffer around each school
    school_buffers = schools.withColumn(
        "buffer_geom",
        GA.ST_Buffer(
            GA.ST_Point("school_lon", "school_lat", 4326),
            300,
            "meters"
        )
    )

    # Spatial join: crashes that fall inside a school buffer
    crashes_in_school_zone = GA.st_join(
        silver, school_buffers,
        spatial_relationship="intersects"
    )

    # Tag crashes: near_school = 1 if inside any school buffer, else 0
    school_zone_dr_nos = (
        crashes_in_school_zone
        .select("dr_no")
        .distinct()
    )

    silver = (
        silver
        .join(school_zone_dr_nos, on="dr_no", how="left")
        .withColumn(
            "near_school",
            F.when(
                school_zone_dr_nos["dr_no"].isNotNull(), F.lit(1)
            ).otherwise(F.lit(0))
        )
    )

    # The left join may duplicate the dr_no column; keep only one
    # (the join key is already resolved by Spark, but drop any extra if present)
    near_school_count = silver.filter(F.col("near_school") == 1).count()
    print(f"Crashes in school zones (300 m buffer): {near_school_count:,}")

except Exception as e:
    print(f"School-zone enrichment skipped: {e}")
    silver = silver.withColumn("near_school", F.lit(0))

In [ ]:
# ── Write silver_crashes Delta table ──────────────────────────────────────────

(
    silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("crash_year")
    .saveAsTable(f"silver_crashes")
)

final_count = spark.read.format("delta").table(f"silver_crashes").count()
print(f"Saved {final_count:,} rows to silver_crashes")

In [ ]:
# ── Enrichment summary ────────────────────────────────────────────────────────

sv = spark.read.format("delta").table(f"silver_crashes")

print("=" * 60)
print("Silver Enrichment Summary")
print("=" * 60)
print(f"  Total rows           : {sv.count():,}")
print(f"  Distinct H3 res-9    : {sv.select('h3_res9').distinct().count():,}")
print(f"  Distinct H3 res-8    : {sv.select('h3_res8').distinct().count():,}")
print(f"  With geometry        : {sv.filter(F.col('geometry').isNotNull()).count():,}")

try:
    school_zone = sv.filter(F.col("near_school") == 1).count()
    print(f"  Near school (300 m)  : {school_zone:,}")
except Exception:
    pass

print("\nSeverity breakdown:")
sv.groupBy("severity").count().orderBy(F.desc("count")).show()

print("=" * 60)
print("Silver layer complete. Proceed to 03_gold_geoanalytics.")